In [ ]:
# installing needed dependencies on colab
!pip install unsloth torch trl evaluate transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Load necessary libraries for GRPO
import unsloth

from trl import GRPOConfig, GRPOTrainer
from transformers import TrainingArguments,GenerationConfig
from unsloth import is_bfloat16_supported
from unsloth import FastLanguageModel

import torch
from datasets import load_dataset

from evaluate import load
from tqdm import tqdm
import json


print("GRPO libraries loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GRPO libraries loaded successfully!


In questa parte del codice vengono codificate:
- strategie avversario
- payoff matrix
- funzione per costruire il prompt da feedare al LLM model
- ```get_model_action()``` che encode il prompt, lo dà al modello e riceve una response. A quel punto outputs la risposta del modello.
- ```run_episode()``` che runna un episodio completo, iterazione per iterazione e returna una lista di dict {"prompt": str, "completion": str, "reward": float}, uno per round



In [ ]:
import re
from datasets import Dataset

# ── Strategie avversario ──────────────────────────────────────────────────────

def tit_for_tat(history):
    """Copia l'ultima mossa del modello. Prima mossa: coopera."""
    if not history:
        return "action1"
    return history[-1]["model"]   # replica l'ultima mossa dell'agente

def always_defect(history):
    return "action2"

def always_cooperate(history):
    return "action1"

OPPONENT_STRATEGIES = {
    "tit_for_tat": tit_for_tat,
    "always_defect": always_defect,
    "always_cooperate": always_cooperate,
}

PAYOFF = {
    ("action1", "action1"): (3, 3),
    ("action1", "action2"): (0, 4),
    ("action2", "action1"): (4, 0),
    ("action2", "action2"): (1, 1),
}

# ── Costruzione prompt ────────────────────────────────────────────────────────

def build_prompt(history, round_num):
    """Costruisce il prompt per il round corrente dato la storia."""
    base = (
        "You are playing an Iterated Prisoner's Dilemma.\n"
        "Payoff matrix:\n"
        "  action1 vs action1 → you get 3, opponent gets 3\n"
        "  action1 vs action2 → you get 0, opponent gets 4\n"
        "  action2 vs action1 → you get 4, opponent gets 0\n"
        "  action2 vs action2 → you get 1, opponent gets 1\n\n"
    )
    if history:
        hist_str = "\n".join(
            f"  Round {i+1}: you played {h['model']}, opponent played {h['opponent']} → you got {h['model_reward']}"
            for i, h in enumerate(history)
        )
        base += f"History so far:\n{hist_str}\n\n"
    base += f"Round {round_num + 1}. What do you play? Answer with ONLY 'action1' or 'action2'.\nYour answer:"
    return base

# ── Generazione output ────────────────────────────────────────────────────────

def get_model_action(model, tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(inputs, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=16,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=True,
        temperature=0.7,
    )
    completion = tokenizer.decode(output_ids[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

    # parsing robusto
    action = "action2"  # default: defect se non capisce
    if re.search(r"action1", completion, re.IGNORECASE):
        action = "action1"
    elif re.search(r"action2", completion, re.IGNORECASE):
        action = "action2"

    return action, completion

# ── Rollout di un episodio ────────────────────────────────────────────────────

def run_episode(model, tokenizer, opponent_strategy_fn, n_rounds=10):
    """
    Ritorna lista di dict:
      {"prompt": str, "completion": str, "reward": float}
    uno per round.
    """
    history = []
    episode_data = []

    for t in range(n_rounds):
        prompt = build_prompt(history, t)
        model_action, completion = get_model_action(model, tokenizer, prompt)
        opponent_action = opponent_strategy_fn(history)

        model_reward, opp_reward = PAYOFF[(model_action, opponent_action)]

        history.append({
            "model": model_action,
            "opponent": opponent_action,
            "model_reward": model_reward,
        })

        episode_data.append({
            "prompt": prompt,
            "completion": completion,
            "reward": float(model_reward),
        })

    return episode_data

# ── Reward shaping opzionale ──────────────────────────────────────────────────

def apply_return_with_discount(episode_data, gamma=0.99):
    """
    Sostituisce i reward istantanei con i ritorni scontati G_t = sum_{k>=t} gamma^k * r_k.
    Utile per dare al modello una visione a lungo termine.
    """
    T = len(episode_data)
    returns = [0.0] * T
    running = 0.0
    for t in reversed(range(T)):
        running = episode_data[t]["reward"] + gamma * running
        returns[t] = running
    for t in range(T):
        episode_data[t]["reward"] = returns[t]
    return episode_data

Questa parte di codice rappresenta il training loop principale.
1. Per ogni epoca avremo ```N_EPISODES```. In ogni epoca costruiamo un dataset di training via rollout.
2. Definiamo la reward function
3. ```GRPOtrainer.train()``` allena il modello a partire da quel training dataset e quella reward function.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import random

N_OUTER_EPOCHS   = 20    # quante volte aggiorni il modello
N_EPISODES       = 16    # episodi per epoch (più = stime reward più stabili)
N_ROUNDS         = 10    # round per episodio
OPPONENTS        = list(OPPONENT_STRATEGIES.values())

training_args = GRPOConfig(
    max_steps=10,                    # step di GRPO per ogni outer epoch
    per_device_train_batch_size=1,
    num_generations=1,               # 1 perché la completion è già nel dataset
    max_prompt_length=512,
    max_completion_length=32,
    learning_rate=5e-6,
    output_dir="outputs",
    report_to="none",
)

for epoch in range(N_OUTER_EPOCHS):
    print(f"\n=== Outer epoch {epoch+1}/{N_OUTER_EPOCHS} ===")

    # 1. Rollout
    rollout_buffer = []
    for _ in range(N_EPISODES):
        opponent_fn = random.choice(OPPONENTS)
        episode = run_episode(model, tokenizer, opponent_fn, n_rounds=N_ROUNDS)
        episode = apply_return_with_discount(episode, gamma=0.99)
        rollout_buffer.extend(episode)

    print(f"  Collected {len(rollout_buffer)} transitions. "
          f"Mean reward: {sum(d['reward'] for d in rollout_buffer)/len(rollout_buffer):.3f}")

    # 2. Costruisci dataset
    dataset = Dataset.from_list([
        {"prompt": d["prompt"], "completion": d["completion"]}
        for d in rollout_buffer
    ])
    reward_lookup = {d["prompt"]: d["reward"] for d in rollout_buffer}

    # 3. Reward function che usa i reward pre-calcolati
    def reward_fn(completions, prompts, **kwargs):
        return [reward_lookup.get(p, 0.0) for p in prompts]

    # 4. Train
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=reward_fn,
        args=training_args,
        train_dataset=dataset,
    )
    trainer.train()